# LangChain Agents

## 과제 스펙

- 새로운 Jupyter notebook에서 리서치 AI 에이전트를 만들고 커스텀 도구를 부여합니다.
- 에이전트는 다음 작업을 수행할 수 있어야 합니다:
    - Wikipedia에서 검색
    - DuckDuckGo에서 검색
    - 웹사이트의 텍스트를 스크랩하고 추출합니다.
    - 리서치 결과를 .txt 파일에 저장하기
- 다음 쿼리로 에이전트를 실행합니다: "Research about the XZ backdoor" 라는 쿼리로 에이전트를 실행하면, 에이전트는 Wikipedia 또는 DuckDuckGo에서 검색을 시도하고, DuckDuckGo에서 웹사이트를 찾으면 해당 웹사이트에 들어가서 콘텐츠를 추출한 다음 .txt 파일에 조사 내용을 저장하는 것으로 완료해야 합니다.

## 0. Init envs

In [1]:
from config.langchain import setup_langchain_openai_envs

setup_langchain_openai_envs()

## 1. Create agent tools

In [2]:
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import Tool

wikipedia_tool = Tool(
    name="Wikipedia",
    func=WikipediaAPIWrapper().run,
    description="Use this to search for information on Wikipedia."
)
duckduckgo_tool = Tool(
    name="duckduckgo search",
    func=DuckDuckGoSearchRun().run,
    description="Use this to search the web using DuckDuckGo."
)

tools=[wikipedia_tool, duckduckgo_tool]

## 2. Create a model

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.1
)

## 3. Create an agent executor

In [ ]:
from langchain.agents import create_structured_chat_agent, AgentExecutor
from langchain import hub

prompt = hub.pull("hwchase17/structured-chat-agent")

agent = create_structured_chat_agent(
    llm=llm,
    tools=tools,
    prompt=prompt,
)
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

## 4. Create search & save file function

In [5]:
def search_and_save_to_file(query, file_name="files/search_results.txt"):
    # Run agent with the query
    response = agent_executor.invoke({
        "input": query
    })
    
    # Save the results to a text file
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(f"Search query: {query}\n\n")
        f.write(f"Agent Response:\n{response}\n")

    print(f"Results saved to {file_name}")

## 5. Do search & save file

In [6]:
search_and_save_to_file('Research about the XZ backdoor')



> Entering new AgentExecutor chain...
```
{
  "action": "duckduckgo search",
  "action_input": "XZ backdoor"
}
```What Does the Backdoor Do? Malicious code added to xz Utils versions 5.6.0 and 5.6.1 modified the way the software functions. The backdoor manipulated sshd, the executable file used to make remote ... That operation matches the style of the XZ Utils backdoor far more than the cruder supply chain attacks of APT41 or Lazarus, by comparison. "It could very well be someone else," says Aitel. A backdoor was planted in the XZ utils package, a popular open-source compression utility for Linux/UNIX systems, by a threat actor named JiaT75. The backdoor exploited the SSH protocol to gain remote access and control over affected devices. A malicious developer inserted a complex backdoor into the open-source library XZ Utils, which affects many Linux distributions and enables remote code execution. Learn how the backdoor works, how to identify it, and what actions to take to protect y

## 6. Check saved file

In [7]:
with open("files/search_results.txt", "r", encoding="utf-8") as f:
    print(f.read())

Search query: Research about the XZ backdoor

Agent Response:
{'input': 'Research about the XZ backdoor', 'output': "The XZ backdoor is a malicious code inserted into versions 5.6.0 and 5.6.1 of the XZ Utils, a popular open-source compression utility for Linux/UNIX systems. This backdoor, introduced by a threat actor named JiaT75, manipulates the SSH protocol to gain unauthorized remote access and control over affected devices. It interferes with SSH authentication, allowing attackers to execute remote code. The backdoor has impacted widely used Linux distributions, including Red Hat and Debian. To protect systems, it's essential to identify the backdoor and take appropriate actions."}

